# Protein Sequence Tokenization Stretegy Comparative Analysis

This notebook implements the phase-2 Tokenization of animo acids and compare strategies based on Vocab size and

<strong>_out of scope:_</strong>

- compare strategies based on Generative and Classification tasks


In [1]:
import os
import sys
import gc
import json
import time
import traceback
import resource
from pathlib import Path

from collections import Counter
from itertools import chain

import math

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from IPython.display import display
from tqdm.auto import tqdm

from util.file_utils import ensure_directories, iter_dataset
from util.text_similarity import rank_texts
from util.seq_util import inspect_residue_distribution
from tokenizer_module import create_tokenizer
import sentencepiece as spm

In [ ]:
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data/interim/protein_eda/protein_corpus.parquet"
OUTPUT_DIR = REPO_ROOT / "data/interim/protein_tokenization"
FIGURE_OUTPUT_DIR = REPO_ROOT / "results/protein_tokenization/figures"
ARTIFACT_OUTPUT_DIR = REPO_ROOT / "results/protein_tokenization/artifacts"

MAX_SEQ_LENGTH = 512
PARSE_BATCH_SIZE = 2**16
# TOKENIZATION_STRATEGIES = ["Unigram", "BPE", "WordPiece", "words", "pairs", "k-mers", "SentencePiece"]
TOKENIZATION_STRATEGIES = ["BPE"]
MIN_FREQUENCIES = [5]
KMER_SIZES = [3, 5, 6, 7, 9]
VOCAB_SIZES = [512]
RARE_RESIDUE_POLICY = "keep"  #  None replace_with_unk replace_with_nn
SAMPLE_MODE = None  # None  #500

SAVE_ARTIFACTS = True
SAVE_FIGURES = True

TOKENIZER_CONFIGS = (
    [{"name": "words", "requires_training": False}]
    + [{"name": "Unigram", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "BPE", "requires_training": True, "vocab_size": vs, "min_frequency": mf} for vs in VOCAB_SIZES for mf in MIN_FREQUENCIES]
    + [{"name": "WordPiece", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "pairs", "requires_training": False, "pair_mode": "sliding"}]
    + [{"name": "k-mers", "requires_training": False, "k": k, "mode": "sliding"} for k in KMER_SIZES]
    + [{"name": "k-mers", "requires_training": False, "k": k, "mode": "non-overlapping"} for k in KMER_SIZES]
    + [{"name": "SentencePiece", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "TFIDF", "requires_training": True}]
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
ensure_directories(ARTIFACT_OUTPUT_DIR, FIGURE_OUTPUT_DIR, OUTPUT_DIR)

## Parse Corpus

Parse every available `protein.faa` file, preserve provenance, and surface malformed records or headers.


In [3]:
protein_dataset = ds.dataset(str(DATA_PATH), format="parquet")

dataset_schema = pd.DataFrame(
    {
        "column": protein_dataset.schema.names,
        "dtype": [str(t) for t in protein_dataset.schema.types],
    }
)
records_total_count = protein_dataset.count_rows()
print(f"Records: {records_total_count:,}")

Records: 88,497,376


In [4]:
if SAMPLE_MODE is not None:
    print(f"Taking a sample of {SAMPLE_MODE} records")
    protein_dataset = ds.dataset(protein_dataset.head(SAMPLE_MODE))
    records_total_count = protein_dataset.count_rows()


display(dataset_schema)

,column,dtype
0,assembly_id,string
1,sequence_id,string
2,ambiguous_residues,string
3,functional_annotations,string
4,raw_header,string
5,sequence_length,int64
6,sequence,string


In [5]:
TOTAL_RECORDS = records_total_count
TOTAL_BATCHES = math.ceil(TOTAL_RECORDS / PARSE_BATCH_SIZE)
print(f"Total batches (batch size {PARSE_BATCH_SIZE}): {TOTAL_BATCHES:,}")

Total batches (batch size 65536): 1,351


In [6]:
STANDARD_RESIDUES = set("ACDEFGHIKLMNPQRSTVWY")
RARE_RESIDUES = set("XBZJUO")  # misstyped tokens that appear in the dataset
VALID_RESIDUES = STANDARD_RESIDUES | RARE_RESIDUES

SPECIAL_TOKENS = [
    "[PAD]",  # padding
    "[UNK]",  # unknown token for out-of-vocabulary residues
    "[CLS]",  # start of sequence (classification token)
    "[SEP]",  # separator token (not used in single sequence tasks but reserved for potential future use)
    "[MASK]",  # mask token for masked language modeling
    "[UNKAA]",  # optional custom placeholder token for rare residues (if using replace_with_unk policy)
]

check correct resudue and distribution


In [7]:
for strategy in TOKENIZATION_STRATEGIES:
    tokenizer_configs = [conf for conf in TOKENIZER_CONFIGS if conf["name"] == strategy]
    # residue_counts = inspect_residue_distribution(protein_dataset, batch_size=PARSE_BATCH_SIZE)

    for config_index, config in enumerate(tokenizer_configs, start=1):
        name = config["name"]
        save_dir = ARTIFACT_OUTPUT_DIR / f"tokenizers/{name}/vocab{config.get('vocab_size', 'default')}_{RARE_RESIDUE_POLICY}"

        tok = create_tokenizer(max_seq_length=MAX_SEQ_LENGTH, rare_residue_policy=RARE_RESIDUE_POLICY, **config)
        print(f"Tokenizer '{name}' config: {config} saved", flush=True)

        if config["requires_training"]:
            tok.train(
                protein_table=protein_dataset,
                save_dir=save_dir,
                batch_size=PARSE_BATCH_SIZE,
                total_batches=TOTAL_BATCHES,
                **{k: v for k, v in config.items() if k not in ("name", "requires_training", "pair_mode")},
            )
        elif name in ("k-mers", "pairs"):
            tok.build_vocab(protein_dataset, batch_size=PARSE_BATCH_SIZE, total_batches=TOTAL_BATCHES)

        tok.save(save_dir)

        learned_tokens = tok.count_learned_tokens()

Tokenizer 'BPE' config: {'name': 'BPE', 'requires_training': True, 'vocab_size': 512, 'min_frequency': 5} saved
[BPETokenizer.train] start vocab_size=512 min_frequency=5 batch_size=65536
[BPETokenizer.train] tokenizer instance created
[BPETokenizer.train] trainer configured
[BPETokenizer.train] writing normalized corpus begin


Preparing BPE corpus:   0%|          | 0.00/88.5M [00:00<?, ?rows/s]

[ProteinTokenizer.write_normalized_corpus] wrote 1770 files to /home/ubuntu/projects/biodata/DNA-BERT/results/protein_tokenization/artifacts/tokenizers/BPE/vocab512_keep/normalized_corpus
[BPETokenizer.train] writing normalized corpus done (1770 files)
[BPETokenizer.train] file training begin



[BPETokenizer.train] file training done
[BPETokenizer.train] postprocessing configuration begin
[BPETokenizer.train] postprocessing configuration done
Vocab size learned: 512
[ProteinTokenizer.save] writing tokenizer JSON to /home/ubuntu/projects/biodata/DNA-BERT/results/protein_tokenization/artifacts/tokenizers/BPE/vocab512_keep/tokenizer.json
Saved BPE tokenizer to: /home/ubuntu/projects/biodata/DNA-BERT/results/protein_tokenization/artifacts/tokenizers/BPE/vocab512_keep/tokenizer.json
[ProteinTokenizer.count_learned_tokens] reading vocab for BPE
Learned tokens: 481


In [8]:
learned_tokens = tok.count_learned_tokens()


[ProteinTokenizer.count_learned_tokens] reading vocab for BPE
Learned tokens: 481
